# APIM ❤️ existing Microsoft Foundry

## Built-in logging lab — existing Foundry deployments

![flow](../../images/built-in-logging.gif)

This version of the lab keeps the existing `jacwang-foundryonly` resource and its model deployments. It deploys only API Management, Application Insights, and Log Analytics for the lab, then grants the APIM managed identity **Cognitive Services User** on the existing Foundry resource.

The Foundry endpoint remains keyless. The `api-key` used in client examples is an APIM subscription key for the gateway, not a Foundry resource key. APIM authenticates to Foundry with managed identity.

### What you will do

1. Verify the Azure subscription, existing resource, and deployment names.
2. Deploy the logging gateway without creating Foundry models.
3. Send requests through the OpenAI-compatible Responses API.
4. Query token usage, prompts, and completions in Log Analytics.

### Prerequisites

- Run `uv sync` from the repository root.
- Sign in with Azure CLI as an identity that can deploy resources and create role assignments.
- Use Python 3.12+ and the VS Code Jupyter extension.


<a id='initialize'></a>
### 0️⃣ Initialize notebook variables

The values below target your existing Foundry resource and its OpenAI-compatible endpoint. Change `model_deployment_name` to select another deployment that supports the Responses API.


In [ ]:
import json, os, sys
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = "built-in-logging-existing-foundry"
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "eastus"

existing_foundry_subscription_id = "6025ba02-1dfd-407f-b358-88f811c7c7aa"
existing_foundry_resource_group_name = "jacwang-rg"
existing_foundry_name = "jacwang-foundryonly"
existing_foundry_endpoint = "https://jacwang-foundryonly.services.ai.azure.com/"
model_deployment_name = "gpt-5-mini"

apim_sku = "Basicv2"
apim_subscriptions_config = [
    {"name": "subscription1", "displayName": "Subscription 1"},
    {"name": "subscription2", "displayName": "Subscription 2"},
    {"name": "subscription3", "displayName": "Subscription 3"},
]

inference_api_path = "inference"
inference_api_type = "AzureOpenAIV1"

utils.print_ok('Notebook initialized')


<a id='verify'></a>
### 1️⃣ Verify Azure CLI and discover the existing deployments

This step selects the requested subscription, verifies that local authentication is disabled, and lists the deployment names already present in `jacwang-foundryonly`. It does not modify the Foundry resource.


In [ ]:
output = utils.run(
    f"az account set --subscription {existing_foundry_subscription_id}",
    "Selected the requested Azure subscription",
    "Failed to select the requested Azure subscription",
)
if not output.success:
    raise RuntimeError(output.text)

output = utils.run(
    "az account show --output json",
    "Retrieved the current Azure account",
    "Failed to get the current Azure account",
)
if not output.success or not output.json_data:
    raise RuntimeError(output.text)

subscription_id = output.json_data['id']
utils.print_info(f"Current user: {output.json_data['user']['name']}")
utils.print_info(f"Subscription: {output.json_data['name']} ({subscription_id})")

if subscription_id.lower() != existing_foundry_subscription_id.lower():
    raise RuntimeError('The active Azure subscription does not match the configured Foundry subscription.')


In [ ]:
foundry_command = (
    f"az cognitiveservices account show --subscription {existing_foundry_subscription_id} "
    f"--resource-group {existing_foundry_resource_group_name} --name {existing_foundry_name} --output json"
)
output = utils.run(foundry_command, "Retrieved the existing Foundry resource", "Failed to retrieve the existing Foundry resource")
if not output.success or not output.json_data:
    raise RuntimeError(output.text)

foundry_resource = output.json_data
foundry = {
    'id': foundry_resource['id'],
    'kind': foundry_resource['kind'],
    'location': foundry_resource['location'],
    'disableLocalAuth': foundry_resource['properties']['disableLocalAuth'],
    'modelInferenceEndpoint': foundry_resource['properties']['endpoints']['Azure AI Model Inference API'],
}
display(foundry)

actual_endpoint = foundry['modelInferenceEndpoint'].rstrip('/') + '/'
if actual_endpoint != existing_foundry_endpoint:
    raise RuntimeError(f"Configured endpoint {existing_foundry_endpoint} does not match {actual_endpoint}.")
if foundry['disableLocalAuth'] is not True:
    utils.print_warning('Local authentication is enabled; this notebook will still use managed identity.')


In [ ]:
import pandas as pd

deployment_command = (
    f"az cognitiveservices account deployment list --subscription {existing_foundry_subscription_id} "
    f"--resource-group {existing_foundry_resource_group_name} --name {existing_foundry_name} "
    "--query \"[].{name:name,model:properties.model.name,format:properties.model.format,"
    "version:properties.model.version,sku:sku.name,state:properties.provisioningState}\" --output json"
)
output = utils.run(deployment_command, "Listed existing model deployments", "Failed to list existing model deployments")
if not output.success:
    raise RuntimeError(output.text)

existing_deployments = output.json_data
display(pd.DataFrame(existing_deployments))

ready_deployment_names = {
    item['name'] for item in existing_deployments if item.get('state') == 'Succeeded'
}
if model_deployment_name not in ready_deployment_names:
    raise RuntimeError(f"Deployment {model_deployment_name!r} is not in a Succeeded state.")
utils.print_ok(f"Using existing deployment: {model_deployment_name}")


<a id='deploy'></a>
### 2️⃣ Deploy the logging gateway with Bicep

[`main-existing-foundry.bicep`](main-existing-foundry.bicep) deploys APIM, Application Insights, and Log Analytics. It references the existing Foundry resource and creates one role assignment so APIM can authenticate with managed identity. It does not create, update, or delete model deployments.


In [ ]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "existingFoundryName": {"value": existing_foundry_name},
        "existingFoundryResourceGroupName": {"value": existing_foundry_resource_group_name},
        "existingFoundrySubscriptionId": {"value": existing_foundry_subscription_id},
        "existingFoundryEndpoint": {"value": existing_foundry_endpoint},
        "inferenceAPIPath": {"value": inference_api_path},
        "inferenceAPIType": {"value": inference_api_type},
    },
}

parameters_path = "params-existing-foundry.json"
with open(parameters_path, "w", encoding="utf-8") as parameters_file:
    json.dump(bicep_parameters, parameters_file, indent=2)

output = utils.run(
    f"az deployment group create --subscription {existing_foundry_subscription_id} "
    f"--name {deployment_name} --resource-group {resource_group_name} "
    f"--template-file main-existing-foundry.bicep --parameters {parameters_path}",
    f"Deployment {deployment_name!r} succeeded",
    f"Deployment {deployment_name!r} failed",
)
if not output.success:
    raise RuntimeError(output.text)


<a id='outputs'></a>
### 3️⃣ Get the deployment outputs

Retrieve the APIM gateway URL, APIM subscription keys, and Log Analytics workspace ID used by the remaining cells.


In [ ]:
output = utils.run(
    f"az deployment group show --subscription {existing_foundry_subscription_id} "
    f"--name {deployment_name} --resource-group {resource_group_name} --output json",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}",
)
if not output.success or not output.json_data:
    raise RuntimeError(output.text)

log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics workspace ID')
apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM service ID')
apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM gateway URL')
foundry_id = utils.get_deployment_output(output, 'existingFoundryId', 'Existing Foundry resource ID')
foundry_role_assignment_id = utils.get_deployment_output(output, 'existingFoundryRoleAssignmentId')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))

for subscription in apim_subscriptions:
    utils.print_info(f"{subscription['displayName']}: ****{subscription['key'][-4:]}")
api_key = apim_subscriptions[0]['key']


<a id='requests'></a>
### 🧪 Test the Responses API with direct HTTP calls

Requests enter APIM at `/inference/openai/v1/responses` and are forwarded to `https://jacwang-foundryonly.services.ai.azure.com/openai/v1/responses`. The APIM subscription key identifies the gateway consumer; APIM's managed identity authenticates to Foundry.


In [ ]:
import requests, time

runs = 3
sleep_time_ms = 100
url = f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1/responses"
payload = {
    "model": model_deployment_name,
    "instructions": "You are a concise, helpful assistant.",
    "input": "What time zone is Seattle in?",
}

with requests.Session() as session:
    session.headers.update({"api-key": api_key, "x-user-id": "alex"})
    for run_number in range(1, runs + 1):
        started = time.perf_counter()
        response = session.post(url, json=payload, timeout=120)
        elapsed = time.perf_counter() - started
        utils.print_response_code(response)
        print(f"▶️ Run {run_number}/{runs}: {elapsed:.2f}s")
        print({key: value for key, value in response.headers.items() if key.lower().startswith('x-ms-')})

        if response.ok:
            data = response.json()
            print(f"Model: {data.get('model')}")
            print(f"Token usage: {json.dumps(data.get('usage', {}), indent=2)}")
            output_text = ''.join(
                content.get('text', '')
                for item in data.get('output', [])
                if item.get('type') == 'message'
                for content in item.get('content', [])
                if content.get('type') == 'output_text'
            )
            print(f"💬 {output_text}\n")
        else:
            print(f"{response.text}\n")

        time.sleep(sleep_time_ms / 1000)


<a id='streaming'></a>
### 🧪 Stream a response with the OpenAI SDK

The OpenAI client is pointed at the APIM `/openai/v1` route. Its `api_key` is the APIM subscription key, not a Foundry key.


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url=f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1",
)

started = time.perf_counter()
chunks = []
try:
    stream = client.responses.create(
        model=model_deployment_name,
        input="Count from 1 to 20, separated by commas.",
        stream=True,
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            chunks.append(event.delta)
            print(event.delta, end="", flush=True)
finally:
    client.close()

print(f"\nCompleted in {time.perf_counter() - started:.2f}s")


<a id='subscriptions'></a>
### 🧪 Generate usage for each APIM subscription

These requests make the per-subscription aggregation visible in the first KQL query. Adjust `runs` if you want a larger sample.


In [ ]:
runs = 3
sleep_time_ms = 100
responses_base_url = f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1"

for run_number in range(1, runs + 1):
    print(f"▶️ Run {run_number}/{runs}")
    for subscription in apim_subscriptions:
        client = OpenAI(
            api_key=subscription['key'],
            base_url=responses_base_url,
        )
        try:
            response = client.responses.create(
                model=model_deployment_name,
                input="Name one benefit of an API gateway in one sentence.",
            )
            print(f"💬 {subscription['displayName']}: {response.output_text}")
        finally:
            client.close()
    print()
    time.sleep(sleep_time_ms / 1000)


<a id='usage'></a>
### 🔍 Display model usage by APIM subscription

Azure Monitor ingestion can take several minutes. If the result is empty immediately after generating traffic, wait briefly and rerun this cell.


In [ ]:
query = """
let llmLogs = ApiManagementGatewayLlmLog
| where TimeGenerated > ago(1h)
| where isnotempty(DeploymentName)
| summarize arg_max(TimeGenerated, *) by CorrelationId, RequestId, SequenceNumber;
let subscriptionLogs = ApiManagementGatewayLogs
| where TimeGenerated > ago(1h)
| summarize arg_max(TimeGenerated, ApimSubscriptionId) by CorrelationId
| project CorrelationId, SubscriptionId = ApimSubscriptionId;
llmLogs
| join kind=leftouter subscriptionLogs on CorrelationId
| extend SubscriptionId = iff(isempty(SubscriptionId), '(no subscription)', SubscriptionId)
| extend ModelName = iff(isempty(ModelName), DeploymentName, ModelName)
| summarize
    LlmCalls = count(),
    MeteredCalls = countif(TotalTokens > 0),
    ZeroTokenCalls = countif(TotalTokens == 0),
    StreamingCalls = countif(IsStreamCompletion),
    SumPromptTokens = sum(PromptTokens),
    SumCompletionTokens = sum(CompletionTokens),
    SumTotalTokens = sum(TotalTokens),
    AvgPromptTokens = round(avgif(todouble(PromptTokens), TotalTokens > 0), 2),
    AvgCompletionTokens = round(avgif(todouble(CompletionTokens), TotalTokens > 0), 2),
    AvgTotalTokens = round(avgif(todouble(TotalTokens), TotalTokens > 0), 2),
    P95TotalTokens = percentile(TotalTokens, 95),
    MaxTotalTokens = max(TotalTokens)
    by SubscriptionId, DeploymentName, ModelName
| extend
    PromptTokenPercent = iff(SumTotalTokens > 0, round(100.0 * todouble(SumPromptTokens) / todouble(SumTotalTokens), 2), 0.0),
    CompletionTokenPercent = iff(SumTotalTokens > 0, round(100.0 * todouble(SumCompletionTokens) / todouble(SumTotalTokens), 2), 0.0)
| order by SumTotalTokens desc
"""

output = utils.run(
    f"az monitor log-analytics query --workspace {log_analytics_id} --analytics-query \"{query}\" --output json",
    "Retrieved model usage logs",
    "Failed to retrieve model usage logs",
)
if output.success:
    display(pd.DataFrame(output.json_data))


<a id='messages'></a>
### 🔍 Display prompts and completions

> Prompt and completion logging can contain sensitive data. Use this setting only where your privacy, compliance, retention, and access-control requirements permit it.


In [ ]:
query = """
ApiManagementGatewayLlmLog
| where TimeGenerated > ago(1h)
| extend RequestArray = parse_json(RequestMessages)
| extend ResponseArray = parse_json(ResponseMessages)
| mv-expand RequestArray
| mv-expand ResponseArray
| project CorrelationId, RequestContent = tostring(RequestArray.content), ResponseContent = tostring(ResponseArray.content), ModelName, TotalTokens, DeploymentName, TimeGenerated
| summarize Input = strcat_array(make_list(RequestContent), ' '), Output = strcat_array(make_list(ResponseContent), ' '), ModelName = take_any(ModelName), TotalTokens = sum(TotalTokens), DeploymentName = take_any(DeploymentName), TimeGenerated = max(TimeGenerated) by CorrelationId
| where isnotempty(Input) and isnotempty(Output)
| order by TimeGenerated desc
"""

output = utils.run(
    f"az monitor log-analytics query --workspace {log_analytics_id} --analytics-query \"{query}\" --output json",
    "Retrieved prompt and completion logs",
    "Failed to retrieve prompt and completion logs",
)
if output.success:
    display(pd.DataFrame(output.json_data))


### Optional exercise: switch deployments

Choose another successful deployment discovered in step 1, assign its deployment name below, then rerun the HTTP/SDK and KQL cells. Avoid the embedding deployment and any model that does not support the Responses API.


In [ ]:
chat_deployment_candidates = [
    item['name']
    for item in existing_deployments
    if item.get('state') == 'Succeeded' and 'embedding' not in item.get('model', '').lower()
]
print('Available candidates:', chat_deployment_candidates)

# Answer scaffold:
# model_deployment_name = chat_deployment_candidates[1]


<a id='cleanup'></a>
### 🗑️ Clean up lab resources

The lab resources are isolated in `lab-built-in-logging-existing-foundry`. Deleting that group removes APIM, Application Insights, and Log Analytics. It does **not** delete `jacwang-foundryonly` or any of its model deployments. Remove the APIM role assignment first so it is not left behind on the existing resource.

Uncomment the next cell only when you intend to remove the lab resources.


In [ ]:
# utils.run(
#     f"az role assignment delete --ids {foundry_role_assignment_id}",
#     "Removed the APIM role assignment from the existing Foundry resource",
#     "Failed to remove the APIM role assignment",
# )
# utils.cleanup_resources(deployment_name, resource_group_name)
